# Task 6 — House Price Prediction + Interest-Rate Slider
Uses the Ames House Prices dataset via OpenML, a preprocessing pipeline and linear Ridge regression. The interest-rate slider is a **scenario simulator**, not a causal claim: Ames does not include mortgage-rate history, so the slider applies a clearly stated sensitivity assumption to the model's baseline prediction.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
ART = Path("artifacts"); FIG = Path("figures"); EX = Path("examples")
for p in [ART, FIG, EX]:
    p.mkdir(exist_ok=True)
print("Folders ready:", ART, FIG, EX)

Folders ready: artifacts figures examples


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/data-doctors/kaggle-house-prices-advanced-regression-techniques/master/data/train.csv"
LOCAL_FALLBACK = Path("train.csv")

def load_ames() -> pd.DataFrame:
    try:
        df = pd.read_csv(DATA_URL)
        print(f"Loaded dataset from GitHub mirror: {df.shape}")
        return df
    except Exception as e:
        print(f"Could not fetch from URL ({e}); trying local fallback '{LOCAL_FALLBACK}'...")
        if LOCAL_FALLBACK.exists():
            df = pd.read_csv(LOCAL_FALLBACK)
            print(f"Loaded dataset from local file: {df.shape}")
            return df
        raise RuntimeError(
            "Dataset could not be loaded from the URL or a local file. "
            "Download the Ames/Kaggle 'House Prices: Advanced Regression Techniques' "
            f"train.csv manually and place it next to this script as '{LOCAL_FALLBACK}'."
        )

df = load_ames()

if "Id" in df.columns:
    df = df.drop(columns="Id")
print(df.shape, df["SalePrice"].describe())

Loaded dataset from GitHub mirror: (1460, 81)
(1460, 80) count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

X = df.drop(columns="SalePrice")
y = df["SalePrice"].astype(float)
num = X.select_dtypes(include=np.number).columns
cat = X.columns.difference(num)

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
])
model = Pipeline([("pre", pre), ("reg", Ridge(alpha=10.0))])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.20, random_state=42)
model.fit(Xtr, ytr)
pred = model.predict(Xte)

rmse = mean_squared_error(yte, pred) ** .5
mae = mean_absolute_error(yte, pred)
r2 = r2_score(yte, pred)
print({"RMSE": rmse, "MAE": mae, "R2": r2})

joblib.dump(model, ART / "house_price_model.joblib")

res = yte - pred
plt.scatter(pred, res, s=12, alpha=.5)
plt.axhline(0, color="black", lw=1)
plt.xlabel("Predicted"); plt.ylabel("Residual")
plt.tight_layout()
plt.savefig(FIG / "residual_plot.png", dpi=160)
plt.close()

sample = pd.DataFrame({"actual": yte.iloc[:5].values, "predicted": pred[:5]})
sample["absolute_error"] = (sample.actual - sample.predicted).abs()
print(sample)
sample.to_csv(EX / "sample_predictions.csv", index=False)

{'RMSE': 30646.4352395474, 'MAE': 19040.572921981628, 'R2': 0.877553578880161}
     actual      predicted  absolute_error
0  154500.0  156307.386529     1807.386529
1  325000.0  337693.728795    12693.728795
2  115000.0   97858.204097    17141.795903
3  159000.0  177185.823295    18185.823295
4  315500.0  337994.670882    22494.670882


In [ ]:
baseline = float(np.median(pred))

def in_jupyter_kernel() -> bool:
    try:
        from IPython import get_ipython
        ip = get_ipython()
        return ip is not None and ip.__class__.__name__ == "ZMQInteractiveShell"
    except Exception:
        return False

def rate_scenario(rate_pp: float) -> float:
    """Apply the stated -6%-per-percentage-point sensitivity assumption."""
    return baseline * ((1 - 0.06) ** rate_pp)

if in_jupyter_kernel():
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    slider = widgets.FloatSlider(value=0, min=0, max=5, step=.25, description="Rate +pp")
    out = widgets.Output()

    def update(change=None):
        with out:
            clear_output(wait=True)
            rate = slider.value
            adjusted = rate_scenario(rate)
            xs = np.linspace(0, 5, 21)
            ys = baseline * ((1 - 0.06) ** xs)
            plt.figure(figsize=(6, 3))
            plt.plot(xs, ys)
            plt.scatter([rate], [adjusted])
            plt.xlabel("Interest-rate increase (percentage points)")
            plt.ylabel("Scenario price")
            plt.show()
            print(f"Baseline: ${baseline:,.0f} | Scenario: ${adjusted:,.0f}")

    slider.observe(update, names="value")
    display(slider, out)
    update()
else:
    print("\n(No Jupyter kernel detected — showing the rate-scenario curve as a static "
          "chart and table instead of the interactive slider.)")
    xs = np.linspace(0, 5, 21)
    ys = baseline * ((1 - 0.06) ** xs)
    plt.figure(figsize=(6, 3))
    plt.plot(xs, ys)
    plt.xlabel("Interest-rate increase (percentage points)")
    plt.ylabel("Scenario price")
    plt.tight_layout()
    plt.savefig(FIG / "rate_scenario_curve.png", dpi=160)
    plt.close()

    scenario_table = pd.DataFrame({"rate_increase_pp": xs, "scenario_price": ys})
    scenario_table.to_csv(EX / "rate_scenario_table.csv", index=False)
    print(scenario_table.round(0).to_string(index=False))
    print(f"\nBaseline: ${baseline:,.0f} | Example at +1.0pp: ${rate_scenario(1.0):,.0f}")
    print("Run this script inside Jupyter/Colab to get the live slider instead.")


(No Jupyter kernel detected — showing the rate-scenario curve as a static chart and table instead of the interactive slider.)
 rate_increase_pp  scenario_price
              0.0        154747.0
              0.0        152371.0
              0.0        150032.0
              1.0        147729.0
              1.0        145462.0
              1.0        143229.0
              2.0        141030.0
              2.0        138866.0
              2.0        136734.0
              2.0        134635.0
              2.0        132569.0
              3.0        130534.0
              3.0        128530.0
              3.0        126557.0
              4.0        124614.0
              4.0        122702.0
              4.0        120818.0
              4.0        118964.0
              4.0        117138.0
              5.0        115340.0
              5.0        113569.0

Baseline: $154,747 | Example at +1.0pp: $145,462
Run this script inside Jupyter/Colab to get the live slider instead.
